# Human 배치·조작 연결 점검
이번 질문은 **보드·사람 선택 배치·조작 순서를 같은 원본 구간으로 연결할 수 있는가?**입니다.
모델 학습·예측 성능·게임 replay를 수행하지 않습니다.

[연결 계약](../../../docs/ai-ml/contracts/human-placement-action-link.md) · [실험 명세](../../../ai-harness/ml-experiments/experiments/human-placement-action-link-readiness.md)

공통 정의 → 예제/내부 검사 → 가상 Smoke → 명시적으로 켜는 작은 Train 표본 → Full 범위 안내 순서입니다.
기본 모두 실행은 실제 dataset·report·raw를 열지 않습니다. Python 파일을 따로 실행할 필요가 없습니다.


In [ ]:
import copy
import hashlib
import json
import os
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPERIMENT_ID = "human-placement-action-link-readiness"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_action_link_readiness.ipynb"
DATASET_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_dataset.ipynb"
READINESS_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
DEFINITION_IDS = {"reuse", "quality-contract", "pair-extraction"}
EXPECTED_DEFS_SHA = "9f790f75c01f7c7343fe569dbb8a2ba0504ce64eea546e10544bef4c8c362ff2"
EXPECTED_READINESS_SHA = "7ccaebcb5db7c1e5c15067f3bceebc793b3ded2c81ccb414834b31d0959ad8ed"
source_cells = [{"id": c["id"], "source": "".join(c["source"])}
                for c in json.loads(DATASET_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
                if c["cell_type"] == "code" and c["id"] in DEFINITION_IDS]
definition_bytes = (json.dumps(source_cells, ensure_ascii=False, sort_keys=True,
                              allow_nan=False, indent=2) + "\n").encode("utf-8")
assert len(source_cells) == 3 and hashlib.sha256(definition_bytes).hexdigest() == EXPECTED_DEFS_SHA
assert hashlib.sha256(READINESS_NOTEBOOK.read_bytes()).hexdigest() == EXPECTED_READINESS_SHA
dataset = {"__name__": "placement_link_shared"}
for cell in source_cells:
    exec(compile(cell["source"], str(DATASET_NOTEBOOK), "exec"), dataset)
dataset["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "finite", "integer", "json_bytes", "sha256_file",
             "normalize_event", "iter_rows", "extract_pairs", "validate_quality"):
    globals()[name] = dataset[name]
checked_paths = dataset["shared"]["checked_paths"]
publish_report = dataset["shared"]["publish_report"]
reject_constant = dataset["shared"]["reject_constant"]
finite_float = dataset["shared"]["finite_float"]

SAMPLE_LINES = (118015, 118021, 118023)
MANIFEST_SHA = "c203a1db7b8f2aa7014c565b371088a9614f6abb0df5632e93fe6c807712fc88"
TRAIN_SHA = "c42c01d6fba0eb7128b77c0773a1669a9be620530a2fb99171dcca67ff3fb7c3"
READINESS_REPORT_SHA = "27f2a5dc3dae6143d22b0930987b3cd102e90d07ae1f228deadd0da41db41b4a"
QUALITY_REPORT_SHA = "97388649259414fe2f5119e76ffa57922b3202f85c920540674f0fc494973ff4"
RAW_SHA = "eb64f86731ecb148ebf398773cb8e23cdf71217fc44f3bcb88460299bb092344"
PIECE_TYPES = {"bar", "square", "tree", "ess", "zee", "eleft", "eright"}


## 1. 무엇을 연결하는가
한 segment는 원본의 한 Human placement 기록입니다. DB export의 game/player/piece는 envelope에 있고 보드·배치는 payload에 있으므로 기존 정규화를 먼저 적용합니다.

| 구분 | 사용 값 |
| --- | --- |
| Placement 입력 후보 | 기록된 boardBefore·activePiece |
| Placement 정답 | selectedPlacement의 실제 셀 집합 |
| Control 입력 후보 | 예측 지점까지 완료된 행동 prefix·이미 지난 간격 |
| Control 정답 | 다음 action·그 action까지의 간격 |
| 연결 증거 | source SHA/line, 정규화 identity, game/player/piece, input index |

고정 직전 보드는 과거 모든 행동의 보드가 아닙니다. 이 점검은 **기록 연결**만 확인하고 입력 성공·과거 geometry·도달 가능성은 확인하지 않습니다.


In [ ]:
def physical_key(piece, selected=False):
    """저장된 셀 형식만 검사한다. 회전/충돌/조각 geometry 규칙을 재구현하지 않는다."""
    require(isinstance(piece, dict) and isinstance(piece.get("pieceType"), str)
            and piece["pieceType"] in PIECE_TYPES, "PIECE_INVALID")
    require(all(type(piece.get(k)) is int for k in ("rotation", "row", "col"))
            and piece["rotation"] >= 0, "PIVOT_INVALID")
    cells = piece.get("cells")
    require(isinstance(cells, list) and len(cells) == 4
            and all(isinstance(c, dict) and type(c.get("row")) is int
                    and type(c.get("col")) is int for c in cells), "CELLS_INVALID")
    ordered = sorted((c["row"], c["col"]) for c in cells)
    require(len(set(ordered)) == 4, "DUPLICATE_CELL")
    if selected:
        require(all(0 <= r < 20 and 0 <= c < 10 for r, c in ordered), "SELECTED_CELL_OUTSIDE_BOARD")
    return piece["pieceType"] + "|" + ";".join(f"{r},{c}" for r, c in ordered)


def observed_prefix(inputs, end):
    """현재까지의 정보만 받는다. gap/시간 역전 뒤부터 prefix를 다시 시작한다."""
    prefix, intervals = [], []
    for index in range(end + 1):
        item = inputs[index]
        require(isinstance(item, dict) and item.get("action") in dataset["shared"]["ACTIONS"]
                and integer(item.get("sequence")) and finite(item.get("createdAt")), "PREFIX_INVALID")
        if index:
            previous = inputs[index - 1]
            contiguous = (item["sequence"] == previous["sequence"] + 1
                          and item["pieceIndex"] == previous["pieceIndex"]
                          and item["createdAt"] >= previous["createdAt"])
            if not contiguous:
                prefix, intervals = [], []
            else:
                intervals.append(item["createdAt"] - previous["createdAt"])
        prefix.append(item["action"])
    return {"prefixActions": prefix, "observedIntervalsMs": intervals}


def join_segment(raw, line, reference, timing_rows, source_sha):
    """원본 segment 하나에서 입력과 정답을 분리한 진단 view를 반환한다."""
    identity, event, payload, trust = normalize_event(raw)
    require(reference["sourceLine"] == line and event["eventType"] == "placement_decision"
            and event["gameId"] == reference["gameId"]
            and event["pieceIndex"] == reference["pieceIndex"], "REFERENCE_MISMATCH")
    require(event["schemaVersion"] in (1, 2) and payload.get("source") == "human", "SOURCE_CONTRACT_INVALID")
    require(type(reference.get("playerNumber")) is int and reference["playerNumber"] > 0,
            "PLAYER_BINDING_MISSING")
    board, active, selected = (payload.get(k) for k in ("boardBefore", "activePiece", "selectedPlacement"))
    require(isinstance(board, list) and len(board) == 20 and all(
        isinstance(row, list) and len(row) == 10 and all(
            c is None or isinstance(c, str) and bool(c.strip()) for c in row) for row in board), "BOARD_INVALID")
    physical_key(active)
    label = physical_key(selected, selected=True)
    require(active["pieceType"] == selected["pieceType"], "PIECE_TYPE_MISMATCH")
    inputs = payload.get("inputSequence")
    require(isinstance(inputs, list) and len(inputs) >= 2
            and all(isinstance(item, dict) for item in inputs), "INPUT_SEQUENCE_INVALID")
    require(all(isinstance(item.get("action"), str)
                and item["action"] in dataset["shared"]["ACTIONS"]
                and all(integer(item.get(k)) for k in ("pieceIndex", "sequence", "tick"))
                and finite(item.get("createdAt")) and item["createdAt"] >= 0 for item in inputs),
            "INPUT_RECORD_INVALID")
    require(not any(item.get("action") == "hold" for item in inputs), "HOLD_BOUNDARY_UNVERIFIED")
    meta = {**event, "line": line}
    pairs, rejected = extract_pairs(inputs, meta, reference["playerNumber"])
    expected = sorted(timing_rows, key=lambda r: r["currentInputIndex"])
    actual = sorted(({**row, "split": "train"} for row in pairs), key=lambda r: r["currentInputIndex"])
    require(expected and actual == expected and len({r["sampleId"] for r in expected}) == len(expected),
            "TIMING_PAIR_MISMATCH")
    control = [{"features": observed_prefix(inputs, row["previousInputIndex"]),
                "labels": {"nextAction": row["nextActionDiagnostic"], "intervalMs": row["recordedIntervalMs"]},
                "currentInputIndex": row["currentInputIndex"]} for row in expected]
    segment_id = hashlib.sha256(json_bytes([source_sha, list(identity), line])).hexdigest()
    return {"segmentId": segment_id, "sourceLine": line, "gameId": event["gameId"],
            "playerNumber": reference["playerNumber"], "pieceIndex": event["pieceIndex"],
            "structuralJoin": "structural_join_verified", "playerMapping": "frozen_readiness_binding",
            "observation": "unverified", "observationEvidence": "historical_build_binding_unverified",
            "reachability": "not_checked", "geometry": "stored_cells_shape_only",
            "actionState": "sequence_only", "trust": trust,
            "negativePivot": selected["col"] < 0 or selected["row"] < 0,
            "placement": {"features": {"boardBefore": copy.deepcopy(board), "activePiece": copy.deepcopy(active)},
                          "labels": {"physicalClassKey": label, "selectedCells": copy.deepcopy(selected["cells"])}},
            "control": control, "pairRejections": dict(sorted(Counter(
                reason for item in rejected for reason in item["reasons"]).items()))}


def audit_records(records, references, rows_by_line, source_sha):
    """불일치는 사유로 기록하며 원본이나 기존 pair를 수정하지 않는다."""
    segments, rejected, seen = [], [], {}
    for line, raw in sorted(records.items()):
        try:
            identity, _, _, _ = normalize_event(raw)
            digest = hashlib.sha256(json_bytes(raw)).hexdigest()
            if identity in seen:
                require(seen[identity] == digest, "IDENTITY_CONFLICT")
                rejected.append({"sourceLine": line, "reason": "EXACT_DUPLICATE_IGNORED"})
                continue
            seen[identity] = digest
            segments.append(join_segment(raw, line, references[line], rows_by_line[line], source_sha))
        except ValueError as error:
            rejected.append({"sourceLine": line, "reason": str(error)})
    return {"segments": segments, "rejected": rejected,
            "counts": {"requestedSegments": len(records), "linkedSegments": len(segments),
                       "linkedPairs": sum(len(s["control"]) for s in segments),
                       "physicalClasses": len({s["placement"]["labels"]["physicalClassKey"] for s in segments}),
                       "games": len({s["gameId"] for s in segments}), "quarantinedSegments": len(rejected)}}


## 2. 작은 예제: 마지막이라는 정보를 미리 주지 않기

`left → rotate → hard_drop`에서 첫 예측의 입력은 `[left]`, 정답은 `rotate`입니다.
아직 일어나지 않은 hard_drop이나 전체 길이 3은 입력이 아닙니다.

선택 배치를 시각화할 수는 있지만 그것은 **정답 label**입니다. 실제 전체 정책에서는 배치 모델이 예측한 목표로 별도 평가해야 합니다.


In [ ]:
def synthetic_case():
    piece = {"pieceType": "square", "rotation": 0, "row": 18, "col": -1,
             "cells": [{"row": r, "col": c} for r in (18, 19) for c in (0, 1)]}
    inputs = [{"action": action, "pieceIndex": 1, "sequence": i + 1, "tick": i, "createdAt": t}
              for i, (action, t) in enumerate((("move_left", 100), ("rotate", 100), ("hard_drop", 250)))]
    raw = {"identity_version": "identity-v1", "event_id": "synthetic-event",
           "event_type": "placement_decision", "schema_version": 2, "game_id": "synthetic-train",
           "player_id": "synthetic-player", "piece_index": 1, "trust_level": "client_reported",
           "payload": {"source": "human", "boardBefore": [[None] * 10 for _ in range(20)],
                       "activePiece": copy.deepcopy(piece), "selectedPlacement": piece, "inputSequence": inputs}}
    ref = {"sourceLine": 1, "gameId": "synthetic-train", "pieceIndex": 1, "playerNumber": 1}
    pairs, rejected = extract_pairs(inputs, {"gameId": ref["gameId"], "pieceIndex": 1, "line": 1}, 1)
    assert not rejected
    return {1: raw}, {1: ref}, {1: [{**r, "split": "train"} for r in pairs]}


def synthetic_audit():
    return audit_records(*synthetic_case(), "synthetic-source-not-real-data")


def build_report(audit, scope, provenance):
    require(scope in ("synthetic", "train_sample"), "FULL_COHORT_NOT_AUTHORIZED")
    axes = {key: dict(sorted(Counter(s[key] for s in audit["segments"]).items()))
            for key in ("structuralJoin", "playerMapping", "observation", "reachability", "geometry", "actionState")}
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "status": "placement_action_link_readiness_complete",
            "runMode": "smoke", "auditScope": scope, "fullCohortCovered": False,
            "policyDecisionEligible": False, "canonicalResultEligible": False,
            "modelFitCount": 0, "testFileRead": False, "validationFileRead": False,
            "datasetPublished": False, "productionIntegration": False,
            "stateConditionedTrainingReady": False, "sourceUnchanged": True,
            "sourceNotebookSha256": sha256_file(SOURCE_NOTEBOOK),
            "datasetDefinitionSha256": EXPECTED_DEFS_SHA,
            "readinessNotebookSha256": EXPECTED_READINESS_SHA,
            "inputs": provenance, "coverage": axes, **audit}


def save_report(report, output):
    if output is None:
        return
    path = Path(output)
    if not path.is_absolute():
        path = ROOT / path
    path.parent.mkdir(parents=True, exist_ok=True)
    destination = checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(path), "smoke")[1]
    publish_report(report, destination)


In [ ]:
def self_checks():
    import tempfile
    checks = []
    records, refs, rows = synthetic_case()
    report = synthetic_audit()
    assert report["counts"]["linkedPairs"] == 2 and not report["rejected"]
    segment = report["segments"][0]
    assert segment["control"][0]["features"] == {"prefixActions": ["move_left"], "observedIntervalsMs": []}
    assert segment["control"][0]["labels"]["intervalMs"] == 0
    assert "selectedPlacement" not in segment["placement"]["features"]
    assert "labels" not in segment["control"][0]["features"]
    assert segment["negativePivot"] and segment["reachability"] == "not_checked"
    checks.append("separate_features_labels_zero_time_negative_pivot")

    alias = copy.deepcopy(records[1]["payload"]["selectedPlacement"])
    alias["rotation"] = 3
    alias["col"] = 0
    alias["cells"].reverse()
    assert physical_key(alias, True) == segment["placement"]["labels"]["physicalClassKey"]
    changed = copy.deepcopy(records)
    changed[1]["payload"]["gameId"] = "conflict"
    assert audit_records(changed, refs, rows, "s")["rejected"][0]["reason"] == "ENVELOPE_PAYLOAD_CONFLICT"
    bad = copy.deepcopy(refs); bad[1]["playerNumber"] = 2
    assert audit_records(records, bad, rows, "s")["rejected"][0]["reason"] == "TIMING_PAIR_MISMATCH"
    bad = copy.deepcopy(rows); bad[1][0]["recordedIntervalMs"] = 999
    assert audit_records(records, refs, bad, "s")["rejected"][0]["reason"] == "TIMING_PAIR_MISMATCH"
    checks.append("alias_envelope_player_time_conflicts")

    changed = copy.deepcopy(records)
    changed[1]["payload"]["selectedPlacement"]["cells"][0]["col"] = 2
    alternative = audit_records(changed, refs, rows, "s")["segments"][0]
    assert alternative["placement"]["features"] == segment["placement"]["features"]
    assert alternative["control"][0]["features"] == segment["control"][0]["features"]
    sequence = copy.deepcopy(records[1]["payload"]["inputSequence"])
    sequence[1]["sequence"] = 10
    assert observed_prefix(sequence, 1)["prefixActions"] == ["rotate"]
    assert observed_prefix(sequence, 1)["observedIntervalsMs"] == []
    changed = copy.deepcopy(records); changed[1]["payload"]["inputSequence"][1]["action"] = "hold"
    assert audit_records(changed, refs, rows, "s")["rejected"][0]["reason"] == "HOLD_BOUNDARY_UNVERIFIED"
    duplicates = {1: records[1], 2: copy.deepcopy(records[1])}
    assert audit_records(duplicates, refs, rows, "s")["counts"]["linkedSegments"] == 1
    assert json_bytes(synthetic_audit()) == json_bytes(synthetic_audit())
    checks.append("future_label_gap_hold_duplicates_determinism")

    for mutation in ("duplicate", "outside", "noninteger"):
        invalid = copy.deepcopy(alias)
        if mutation == "duplicate": invalid["cells"][0] = invalid["cells"][1]
        elif mutation == "outside": invalid["cells"][0]["col"] = 10
        else: invalid["cells"][0]["row"] = True
        try:
            physical_key(invalid, True)
        except ValueError:
            pass
        else:
            raise AssertionError("INVALID_CELLS_ACCEPTED")
    checks.append("malformed_cell_rejection")
    malformed = copy.deepcopy(records)
    del malformed[1]["payload"]["inputSequence"][0]["pieceIndex"]
    assert audit_records(malformed, refs, rows, "s")["rejected"][0]["reason"] == "INPUT_RECORD_INVALID"
    malformed = copy.deepcopy(records)
    malformed[1]["payload"]["activePiece"]["pieceType"] = []
    assert audit_records(malformed, refs, rows, "s")["rejected"][0]["reason"] == "PIECE_INVALID"
    checks.append("malformed_input_quarantine")
    with tempfile.TemporaryDirectory() as folder:
        target = Path(folder) / "existing.json"; target.write_bytes(b"preserve")
        hard = Path(folder) / "hard.json"; os.link(target, hard)
        sym = Path(folder) / "sym.json"; sym.symlink_to(target)
        dangling = Path(folder) / "dangling.json"; dangling.symlink_to(Path(folder) / "absent")
        for output in (target, hard, sym, dangling, Path(folder) / "child" / ".." / "existing.json"):
            try: save_report({}, output)
            except (ValueError, FileExistsError): pass
            else: raise AssertionError("OUTPUT_NOT_PROTECTED")
        assert target.read_bytes() == b"preserve"
    checks.append("output_alias_symlink_hardlink")
    return pd.DataFrame({"check": checks, "result": "passed"})


display(self_checks())


## 3. 실제 작은 Train 표본을 읽는 방법

승인된 sourceLine **118015, 118021, 118023**만 raw JSON으로 파싱합니다.
기존 readiness의 playerNumber binding과 품질 report를 재사용하며 전체 player index를 새로 조사하지 않습니다.

원본 bytes 전체는 SHA 확인을 위해 읽습니다. Train timing 파일은 lookup을 위해 순회하므로 **visited rows**와 **실제 연결 pair 수**는 다릅니다.
Validation/Test 파일은 열지 않습니다. 원본의 다른 줄은 위치를 찾으려고 지나가지만 JSON 파싱·평가하지 않습니다.


In [ ]:
def repo_input(name):
    relative = Path(name)
    require(not relative.is_absolute() and ".." not in relative.parts, "INPUT_MUST_BE_REPO_RELATIVE")
    result = (ROOT / relative).resolve(strict=True)
    require(result.is_relative_to(ROOT), "INPUT_OUTSIDE_REPOSITORY")
    return result


def checked_json(path, digest):
    require(sha256_file(path) == digest, "INPUT_SHA_MISMATCH")
    return json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_constant)


def run_train_sample(output=None):
    """지정 Train 세 줄만 연결하고 점검 전후 SHA 일치를 확인해 반환한다."""
    current = [{"id": c["id"], "source": "".join(c["source"])}
               for c in json.loads(DATASET_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
               if c["cell_type"] == "code" and c["id"] in DEFINITION_IDS]
    require(hashlib.sha256(json_bytes(current)).hexdigest() == EXPECTED_DEFS_SHA,
            "DEPENDENCY_DEFINITIONS_CHANGED")
    raw_path = repo_input(os.environ.get("HUMAN_PLACEMENT_LINK_RAW", dataset["DEFAULT_SOURCE"]))
    directory = repo_input(os.environ.get("HUMAN_PLACEMENT_LINK_DATASET",
                                         "data/datasets/human-timing-v1-20261024-full"))
    manifest_path, train_path = directory / "manifest.json", directory / "train.jsonl"
    readiness_path = ROOT / "data/ml-experiments/human-action-data-readiness/full-report.json"
    quality_path = ROOT / "data/ml-experiments/human-game-quality-audit/full-report.json"
    watched = {SOURCE_NOTEBOOK: sha256_file(SOURCE_NOTEBOOK),
               DATASET_NOTEBOOK: sha256_file(DATASET_NOTEBOOK), READINESS_NOTEBOOK: EXPECTED_READINESS_SHA,
               raw_path: RAW_SHA, manifest_path: MANIFEST_SHA, train_path: TRAIN_SHA,
               readiness_path: READINESS_REPORT_SHA, quality_path: QUALITY_REPORT_SHA}
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_SHA_MISMATCH")
    manifest = checked_json(manifest_path, MANIFEST_SHA)
    readiness = checked_json(readiness_path, READINESS_REPORT_SHA)
    quality = checked_json(quality_path, QUALITY_REPORT_SHA)
    require(readiness["source"]["sha256"] == RAW_SHA and readiness["sourceUnchanged"] is True,
            "READINESS_SOURCE_MISMATCH")
    approved = validate_quality(quality, RAW_SHA, raw_path.stat().st_size)
    groups = manifest["split"]["gameIds"]
    require(set(groups) == {"train", "validation", "test"}
            and all(len(groups[s]) == len(set(groups[s])) for s in groups)
            and not any(set(groups[a]) & set(groups[b]) for a, b in
                        (("train", "validation"), ("train", "test"), ("validation", "test"))),
            "GAME_SPLIT_LEAKAGE")
    require(manifest["source"]["sha256"] == RAW_SHA and manifest["files"]["train.jsonl"]["sha256"] == TRAIN_SHA,
            "MANIFEST_SOURCE_MISMATCH")
    references = {}
    for reference in readiness["detail"]["segments"]:
        if reference["sourceLine"] in SAMPLE_LINES:
            require(reference["sourceLine"] not in references, "AMBIGUOUS_REFERENCE")
            game = reference["gameId"]
            require(game in groups["train"] and game in approved
                    and approved[game]["status"] == "include_candidate", "NOT_APPROVED_TRAIN")
            require(reference["playerNumber"] in {p["playerNumber"] for p in approved[game]["humanPlayers"]},
                    "PLAYER_BINDING_MISMATCH")
            references[reference["sourceLine"]] = reference
    require(set(references) == set(SAMPLE_LINES), "REFERENCE_MISSING")
    rows_by_line = {line: [] for line in SAMPLE_LINES}
    visited = 0
    for _, row, kind in iter_rows(train_path):
        visited += 1
        require(kind == "parsed", "TRAIN_PARSE_FAILURE")
        if row["sourceLine"] not in rows_by_line:
            continue
        reference = references[row["sourceLine"]]
        require(row["split"] == "train" and row["gameId"] == reference["gameId"]
                and row["pieceIndex"] == reference["pieceIndex"]
                and row["playerNumber"] == reference["playerNumber"], "TIMING_REFERENCE_MISMATCH")
        rows_by_line[row["sourceLine"]].append(row)
    require(visited == 41151 and all(rows_by_line.values()), "TRAIN_LOOKUP_COUNT_MISMATCH")
    records = {}
    with raw_path.open("rb") as handle:
        for line, data in enumerate(handle, 1):
            if line in references:
                records[line] = json.loads(data, parse_constant=reject_constant, parse_float=finite_float)
            if line == max(SAMPLE_LINES):
                break
    require(set(records) == set(SAMPLE_LINES), "RAW_LINE_MISSING")
    audit = audit_records(records, references, rows_by_line, RAW_SHA)
    report = build_report(audit, "train_sample", {
        "rawSha256": RAW_SHA, "manifestSha256": MANIFEST_SHA, "trainSha256": TRAIN_SHA,
        "readinessReportSha256": READINESS_REPORT_SHA, "qualityReportSha256": QUALITY_REPORT_SHA,
        "selectedSourceLines": list(SAMPLE_LINES), "rawRecordsParsed": len(records),
        "trainTimingRowsVisited": visited})
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    save_report(report, output)
    return report


In [ ]:
def show_link(report):
    """보존된 진단 값만 표와 그림으로 표시하며 학습이나 파일 읽기는 하지 않는다."""
    scope = report["auditScope"]
    print(scope, "|", report["status"], "| model fit=0, full cohort=false")
    display(pd.DataFrame([report["counts"]]))
    display(pd.DataFrame(report["coverage"]).fillna(0).astype(int))
    display(pd.DataFrame(report["rejected"], columns=["sourceLine", "reason"]))
    summaries = [{"sourceLine": s["sourceLine"], "pieceIndex": s["pieceIndex"],
                  "pieceType": s["placement"]["features"]["activePiece"]["pieceType"],
                  "joinedPairs": len(s["control"]), "negativePivot": s["negativePivot"],
                  "playerMapping": s["playerMapping"]} for s in report["segments"]]
    display(pd.DataFrame(summaries))
    examples = [{"sourceLine": s["sourceLine"], "inputIndex": p["currentInputIndex"],
                 "observedPrefix": " → ".join(p["features"]["prefixActions"]),
                 "nextActionLabel": p["labels"]["nextAction"], "intervalLabelMs": p["labels"]["intervalMs"]}
                for s in report["segments"] for p in s["control"]]
    display(pd.DataFrame(examples))
    if not report["segments"]:
        print("No linked segment; inspect quarantine reasons. This is not training-ready.")
        return
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, axis in zip(axes, ("structuralJoin", "observation", "reachability")):
        pd.Series(report["coverage"][axis]).plot.bar(ax=ax, rot=15)
        ax.set_ylabel("sample segment count")
        ax.set_title(axis)
    fig.suptitle(scope + " — independent audit axes")
    plt.tight_layout(); plt.show(); plt.close(fig)
    fig, axes = plt.subplots(1, len(report["segments"]), figsize=(4 * len(report["segments"]), 6), squeeze=False)
    for ax, segment in zip(axes[0], report["segments"]):
        board = segment["placement"]["features"]["boardBefore"]
        ax.imshow([[int(cell is not None) for cell in row] for row in board], cmap="Greys", vmin=0, vmax=1)
        selected = segment["placement"]["labels"]["selectedCells"]
        ax.scatter([c["col"] for c in selected], [c["row"] for c in selected],
                   marker="s", s=65, c="tab:green", label="Human selected cells (label)")
        ax.set_title(f"sourceLine {segment['sourceLine']}\ngeometry/reachability NOT checked")
        ax.set(xlabel="col", ylabel="row", xlim=(-0.5, 9.5), ylim=(19.5, -0.5))
        ax.legend(fontsize=7)
    fig.suptitle(scope + " — recorded board + label overlay, not action-time state")
    plt.tight_layout(); plt.show(); plt.close(fig)
    print("Structural join is not observation/reachability verification or model performance.")


## 4. Smoke — 설정 → 실행 → 결과
가상 한 segment의 두 pair로 실제와 같은 연결 함수를 확인합니다. 실제 게임 성능은 측정하지 않습니다.


In [ ]:
SMOKE_OUTPUT = os.environ.get("HUMAN_PLACEMENT_LINK_SMOKE_REPORT") or None


In [ ]:
SMOKE_REPORT = build_report(synthetic_audit(), "synthetic", {"kind": "synthetic_only"})
save_report(SMOKE_REPORT, SMOKE_OUTPUT)


In [ ]:
show_link(SMOKE_REPORT)


## 5. 작은 Train 표본 — 설정 → 실행 → 결과
실제 입력을 확인하려면 아래 `RUN_TRAIN_SAMPLE=True`로 바꾼 뒤 설정 → 실행 → 결과 셀을 실행합니다.
지정된 세 줄만 검사하며 모델 학습은 없습니다. 출력은 기본 메모리이고 저장하려면 새 JSON 경로를 지정하세요.

기존 파일은 덮어쓰지 않습니다. 저장 위치는 이 실험의 ignored 디렉터리 또는 임시 디렉터리입니다.
이 단계는 **전체 cohort Full이 아닙니다.**


In [ ]:
RUN_TRAIN_SAMPLE = False
SAMPLE_OUTPUT = os.environ.get("HUMAN_PLACEMENT_LINK_SAMPLE_REPORT") or None


In [ ]:
TRAIN_SAMPLE_REPORT = None
if RUN_TRAIN_SAMPLE:
    TRAIN_SAMPLE_REPORT = run_train_sample(SAMPLE_OUTPUT)
else:
    print("Train sample skipped. 실제 입력을 확인하려면 RUN_TRAIN_SAMPLE=True로 설정하세요.")


In [ ]:
if TRAIN_SAMPLE_REPORT is not None:
    show_link(TRAIN_SAMPLE_REPORT)
else:
    print("실제 표본 결과 없음. 위 결과는 가상 Smoke입니다.")


## 6. Full — 이번 범위에서 실행 불가
이번 승인은 Notebook 구현과 작은 Train 표본뿐입니다.
전체 cohort 감사·전체 player index·core replay·dataset 생성·모델 학습은 구현/실행하지 않았습니다.
이전 timing/joint Notebook의 Full과 달리 여기서 `RUN_FULL`을 켜도 전체 처리를 시작하지 않습니다.
향후 Full 범위·cohort·규칙 지원을 별도 명세로 정한 후 구현해야 합니다.


In [ ]:
RUN_FULL = False
require(not RUN_FULL, "FULL_COHORT_REQUIRES_SEPARATE_SPEC_AND_APPROVAL")
print("Full is not implemented for this bounded readiness check.")


## 7. 해석과 다음 gate
- `structural_join_verified`: 같은 원본·정제 pair와 label이 연결됐습니다. 당시 보드가 매 action에 주어졌다는 뜻은 아닙니다.
- `observation=unverified`: 고정 직전 보드와 anchor의 과거 build/시점 대응은 확인하지 못했습니다.
- `reachability=not_checked`: 후보 생성·현재 core·과거 규칙 replay를 실행하지 않았습니다.
- `actionState=sequence_only`: 기록된 시도열이며 성공 경로가 아닙니다.
- `geometry=stored_cells_shape_only`: 저장된 좌표 형식 검사이며 실제 조각 정의·회전의 검증이 아닙니다.

세 실제 표본 모두 연결돼도 전체 데이터/학습 준비 완료로 판단하지 않습니다.
다음은 남은 관측 시점·지원 규칙의 근거를 정리하고 배치 오프라인 과제에 쓸 수 있는 범위를 선택하는 것입니다.
추가 로그 수집·하드드롭 가중치 변경·모델 학습·사이트 연결은 자동 진행하지 않습니다.
